## 1. 기본 패키지 불러오기
전처리와 리포트 저장에 필요한 패키지를 불러옵니다. 실제 경로와 처리 로직은 기존 코드 그대로 유지했습니다.

In [1]:
# 기본 패키지
from pathlib import Path
from datetime import datetime
import json
import os
import warnings

import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("polars:", pl.__version__)
print("pandas:", pd.__version__)


polars: 1.35.2
pandas: 2.2.2


## 2. 팀 공통 Google Drive 경로 설정
원본 데이터 위치와 저장 폴더를 지정합니다. Colab 환경에서 바로 실행되도록 기존 경로를 유지합니다.

In [2]:
from pathlib import Path

# ============================================================
# Colab + Google Drive / Local path setup
# ============================================================
# 팀원 공통 실행 기준:
# - raw 원본 데이터: 기업 Google Drive
# - 큰 parquet / 팀 공유 산출물: 기업 Google Drive
# - 가벼운 검증표 / 피처 명세서 / 요약 CSV: Colab local
#
# Drive 폴더 구조:
# GraphAML/
# └── data/
#     ├── code/
#     ├── raw/
#     ├── processed/
#     │   ├── step01_clean_base/
#     │   └── ml_features/
#     └── graph/
#         └── gnn_inputs/
#
# Local 폴더 구조:
# /content/GraphAML_local_outputs/
# ├── step01_clean_base/
# ├── ml_features/
# └── gnn_inputs/
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    raise RuntimeError(
        "이 노트북은 기업용 Colab + Google Drive 실행 기준입니다. "
        "Colab에서 열고 Google Drive를 마운트한 뒤 실행해주세요."
    )

# Drive: raw 데이터, 큰 parquet, 팀 공유 산출물
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/GraphAML")
DRIVE_DATA_DIR = DRIVE_BASE_DIR / "data"

CODE_DIR = DRIVE_DATA_DIR / "code"
RAW_DIR = DRIVE_DATA_DIR / "raw"
PROCESSED_DIR = DRIVE_DATA_DIR / "processed"
GRAPH_DIR = DRIVE_DATA_DIR / "graph"

STEP01_DRIVE_DIR = PROCESSED_DIR / "step01_clean_base"
ML_DRIVE_DIR = PROCESSED_DIR / "ml_features"
GNN_DRIVE_DIR = GRAPH_DIR / "gnn_inputs"

# Local: 가벼운 검증표, 피처 명세서, 요약 CSV
# 주의: Colab local이므로 런타임 종료 시 사라질 수 있습니다.
LOCAL_BASE_DIR = Path("/content/GraphAML_local_outputs")
STEP01_LOCAL_DIR = LOCAL_BASE_DIR / "step01_clean_base"
ML_LOCAL_DIR = LOCAL_BASE_DIR / "ml_features"
GNN_LOCAL_DIR = LOCAL_BASE_DIR / "gnn_inputs"

# 기존 코드 호환용 alias
BASE_DIR = DRIVE_BASE_DIR
DATA_DIR = DRIVE_DATA_DIR
STEP01_DIR = STEP01_DRIVE_DIR
ML_DIR = ML_DRIVE_DIR
GNN_DIR = GNN_DRIVE_DIR
PROJECT_DIR = DRIVE_BASE_DIR

for p in [
    CODE_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    GRAPH_DIR,
    STEP01_DRIVE_DIR,
    ML_DRIVE_DIR,
    GNN_DRIVE_DIR,
    STEP01_LOCAL_DIR,
    ML_LOCAL_DIR,
    GNN_LOCAL_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_BASE_DIR   :", DRIVE_BASE_DIR)
print("DRIVE_DATA_DIR   :", DRIVE_DATA_DIR)
print("RAW_DIR          :", RAW_DIR)
print("STEP01_DRIVE_DIR :", STEP01_DRIVE_DIR)
print("ML_DRIVE_DIR     :", ML_DRIVE_DIR)
print("GNN_DRIVE_DIR    :", GNN_DRIVE_DIR)
print("LOCAL_BASE_DIR   :", LOCAL_BASE_DIR)
print("STEP01_LOCAL_DIR :", STEP01_LOCAL_DIR)
print("ML_LOCAL_DIR     :", ML_LOCAL_DIR)
print("GNN_LOCAL_DIR    :", GNN_LOCAL_DIR)


# 02번은 raw 거래 파일을 읽고 clean base를 저장합니다.
def resolve_raw_transaction_path(raw_dir):
    raw_dir = Path(raw_dir)
    candidate_names = [
        "HI-Small_Trans.csv", "HI-Small_Trans.parquet",
        "LI-Small_Trans.csv", "LI-Small_Trans.parquet",
        "HI-Medium_Trans.csv", "LI-Medium_Trans.csv",
        "HI-Large_Trans.csv", "LI-Large_Trans.csv",
        "transactions.csv", "Transactions.csv", "trans.csv",
    ]
    for name in candidate_names:
        p = raw_dir / name
        if p.exists():
            return p

    files = sorted([p for p in raw_dir.iterdir() if p.is_file()])
    trans_like = [
        p for p in files
        if p.suffix.lower() in [".csv", ".parquet", ".pq"]
        and ("trans" in p.name.lower() or "transaction" in p.name.lower())
    ]
    if len(trans_like) == 1:
        return trans_like[0]
    if len(trans_like) > 1:
        print("거래 파일 후보가 여러 개입니다. 첫 번째 파일을 사용합니다:")
        for p in trans_like:
            print(" -", p.name)
        return trans_like[0]

    available = "\n".join([f" - {p.name}" for p in files[:50]])
    raise FileNotFoundError(
        "거래 데이터 파일을 자동으로 찾지 못했습니다.\n"
        "GraphAML/data/raw 폴더의 파일명을 확인해주세요.\n"
        f"RAW_DIR: {raw_dir}\n확인된 파일:\n{available}"
    )

RAW_DATA_PATH = resolve_raw_transaction_path(RAW_DIR)
DRIVE_OUTPUT_DIR = STEP01_DRIVE_DIR
LOCAL_OUTPUT_DIR = STEP01_LOCAL_DIR
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "ACCOUNTS_PATH": RAW_DIR / "HI-Small_accounts.csv",       # 참고용. 이번 단계에서는 읽지 않음
    "PATTERNS_PATH": RAW_DIR / "HI-Small_Patterns.txt",       # 참고용. 피처로 사용 금지

    # 큰 산출물 / 다음 단계 입력용은 Drive에 저장
    "DRIVE_OUTPUT_DIR": DRIVE_OUTPUT_DIR,

    # 가벼운 검증표 / 요약 CSV는 Colab local에 저장
    "LOCAL_OUTPUT_DIR": LOCAL_OUTPUT_DIR,

    # 기존 코드 호환용: 02번의 주 산출물 위치는 Drive
    "OUTPUT_DIR": DRIVE_OUTPUT_DIR,
    "SAMPLE_MODE": False,
    "SAMPLE_ROWS": 100_000,
    "INFER_SCHEMA_LENGTH": 10000,

    # 은행 ID + 계좌번호를 묶어서 계좌 ID로 사용
    # 원본 Account 값만 쓰려면 False로 변경
    "USE_BANK_ACCOUNT_COMPOSITE_ID": True,

    "TIMESTAMP_FORMATS": [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
        "%Y-%m-%d %H:%M:%S%.f",
        "%Y/%m/%d %H:%M:%S",
        "%Y/%m/%d %H:%M",
    ],

    "TIMESTAMP_COL_CANDIDATES": [
        "Timestamp", "timestamp", "Time", "time", "DateTime", "datetime"
    ],
    "LABEL_COL_CANDIDATES": [
        "Is Laundering", "Is_Laundering", "is_laundering", "label", "Label", "target"
    ],
    "SENDER_COL_CANDIDATES": [
        "Account", "From Account", "From_Account", "from_account", "sender_account"
    ],
    "RECEIVER_COL_CANDIDATES": [
        "Account.1", "To Account", "To_Account", "to_account", "receiver_account"
    ],
    "FROM_BANK_COL_CANDIDATES": [
        "From Bank", "From_Bank", "from_bank"
    ],
    "TO_BANK_COL_CANDIDATES": [
        "To Bank", "To_Bank", "to_bank"
    ],
    "AMOUNT_COL_CANDIDATES": [
        "Amount Paid", "Amount_Paid", "amount_paid", "Amount Received", "Amount_Received", "amount"
    ],
    "CURRENCY_COL_CANDIDATES": [
        "Payment Currency", "Payment_Currency", "Receiving Currency", "Receiving_Currency", "Currency"
    ],
    "PAYMENT_FORMAT_COL_CANDIDATES": [
        "Payment Format", "Payment_Format", "payment_format", "Payment Type"
    ],
    "TX_ID_COL_CANDIDATES": [
        "tx_id", "transaction_id", "Transaction ID", "Transaction_ID", "id", "ID"
    ],

    "STRICT_LABEL_CHECK": True,
    "RANDOM_SEED": 42,

    # polars 0.20.31 기준.
    "SCHEMA_OVERRIDES": {
        "Timestamp": pl.Utf8,
        "From Bank": pl.Utf8,
        "To Bank": pl.Utf8,
        "Account": pl.Utf8,
        "Account.1": pl.Utf8,
        "Amount Received": pl.Float64,
        "Amount Paid": pl.Float64,
        "Receiving Currency": pl.Utf8,
        "Payment Currency": pl.Utf8,
        "Payment Format": pl.Utf8,
        "Is Laundering": pl.Int64,
    },
}

print("BASE_DIR:", CONFIG["BASE_DIR"])
print("DATA_DIR:", CONFIG["DATA_DIR"])
print("RAW     :", CONFIG["RAW_DATA_PATH"], "exists=", CONFIG["RAW_DATA_PATH"].exists())
print("DRIVE_OUT:", CONFIG["DRIVE_OUTPUT_DIR"])
print("LOCAL_OUT:", CONFIG["LOCAL_OUTPUT_DIR"])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_BASE_DIR   : /content/drive/MyDrive/GraphAML
DRIVE_DATA_DIR   : /content/drive/MyDrive/GraphAML/data
RAW_DIR          : /content/drive/MyDrive/GraphAML/data/raw
STEP01_DRIVE_DIR : /content/drive/MyDrive/GraphAML/data/processed/step01_clean_base
ML_DRIVE_DIR     : /content/drive/MyDrive/GraphAML/data/processed/ml_features
GNN_DRIVE_DIR    : /content/drive/MyDrive/GraphAML/data/graph/gnn_inputs
LOCAL_BASE_DIR   : /content/GraphAML_local_outputs
STEP01_LOCAL_DIR : /content/GraphAML_local_outputs/step01_clean_base
ML_LOCAL_DIR     : /content/GraphAML_local_outputs/ml_features
GNN_LOCAL_DIR    : /content/GraphAML_local_outputs/gnn_inputs
BASE_DIR: /content/drive/MyDrive/GraphAML
DATA_DIR: /content/drive/MyDrive/GraphAML/data
RAW     : /content/drive/MyDrive/GraphAML/data/raw/HI-Small_Trans.csv exists= True
DRIVE_OUT: /content/drive/MyDrive/GraphAML/data/proc

## 3. 공통 유틸 함수 준비
검증 결과, 메모리 사용량, 저장 리포트를 만들 때 반복해서 쓰는 함수입니다. 코드 중복을 줄이고, 문제가 생겼을 때 어디서 발생했는지 추적하기 쉽게 합니다.

In [3]:
# 저장과 검증에 반복해서 쓰는 함수들
# =========================
# 작은 유틸
# =========================

validation_records = []
memory_records = []


def _now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def add_validation(check_name, status, severity, details, n_failed_rows=0):
    validation_records.append({
        "record_type": "validation",
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "details": str(details),
        "n_failed_rows": int(n_failed_rows or 0),
        "created_at": _now(),
    })


def normalize_name(x):
    return str(x).strip().lower().replace("_", " ").replace("-", " ")


def detect_column(columns, candidates, required=True, logical_name="column"):
    norm_to_real = {normalize_name(c): c for c in columns}
    for cand in candidates:
        key = normalize_name(cand)
        if key in norm_to_real:
            return norm_to_real[key]

    if required:
        raise ValueError(
            f"필수 컬럼을 찾지 못했습니다: {logical_name}\n"
            f"후보: {candidates}\n"
            f"현재 컬럼: {list(columns)}"
        )
    return None


def estimate_memory_mb(df):
    try:
        return float(df.estimated_size("mb"))
    except Exception:
        return None


def profile_memory(df, step_name):
    memory_mb = estimate_memory_mb(df)
    memory_records.append({
        "step_name": step_name,
        "n_rows": df.height,
        "n_cols": df.width,
        "memory_mb": memory_mb,
        "created_at": _now(),
    })
    msg = f"[{step_name}] rows={df.height:,}, cols={df.width:,}"
    if memory_mb is not None:
        msg += f", memory={memory_mb:.2f} MB"
    print(msg)


# 한글이 깨지지 않도록 utf-8-sig로 CSV를 저장합니다.
def write_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.write_csv(path)


def compact_col_name(name):
    return str(name).strip().lower().replace(" ", "_").replace(".", "_").replace("-", "_")


## 4. 원본 거래 데이터 로드
원본 CSV를 읽어 공통 clean base를 만들 준비를 합니다. 이 단계부터는 EDA가 아니라 실제 전처리 파이프라인입니다.

In [4]:
# 원본 거래 데이터를 읽는 단계
# =========================
# 데이터 로드
# =========================

# 원본 거래 파일을 읽고, 이후 단계에서 필요한 기본 DataFrame을 준비합니다.
def read_input_data(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"입력 파일이 없습니다: {path}")

    suffix = path.suffix.lower()
    n_rows = CONFIG["SAMPLE_ROWS"] if CONFIG["SAMPLE_MODE"] else None

    if suffix == ".csv":
        # polars 버전별 인자명 차이 대응
        try:
            df = pl.read_csv(
                path,
                infer_schema_length=CONFIG["INFER_SCHEMA_LENGTH"],
                schema_overrides=CONFIG["SCHEMA_OVERRIDES"],
                n_rows=n_rows,
                ignore_errors=False,
            )
        except TypeError:
            df = pl.read_csv(
                path,
                infer_schema_length=CONFIG["INFER_SCHEMA_LENGTH"],
                dtypes=CONFIG["SCHEMA_OVERRIDES"],
                n_rows=n_rows,
                ignore_errors=False,
            )
    elif suffix in [".parquet", ".pq"]:
        if CONFIG["SAMPLE_MODE"]:
            df = pl.scan_parquet(path).limit(CONFIG["SAMPLE_ROWS"]).collect()
        else:
            df = pl.read_parquet(path)
    else:
        raise ValueError(f"지원하지 않는 파일 형식입니다: {suffix}")

    # 원본 순서 보존. timestamp 동률일 때 안정적인 정렬 기준으로 사용.
    df = df.with_row_index("source_row_number")
    return df


df_raw = read_input_data(CONFIG["RAW_DATA_PATH"])
profile_memory(df_raw, "01_loaded_raw")

print(df_raw.columns)
df_raw.head(3)


[01_loaded_raw] rows=5,078,345, cols=12, memory=464.71 MB
['source_row_number', 'Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


source_row_number,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
u32,str,str,str,str,str,f64,str,f64,str,str,i64
0,"""2022/09/01 00:20""","""010""","""8000EBD30""","""010""","""8000EBD30""",3697.34,"""US Dollar""",3697.34,"""US Dollar""","""Reinvestment""",0
1,"""2022/09/01 00:20""","""03208""","""8000F4580""","""001""","""8000F5340""",0.01,"""US Dollar""",0.01,"""US Dollar""","""Cheque""",0
2,"""2022/09/01 00:00""","""03209""","""8000F4670""","""03209""","""8000F4670""",14675.57,"""US Dollar""",14675.57,"""US Dollar""","""Reinvestment""",0


## 5. 컬럼명 표준화와 필수 컬럼 확인
ML/GNN 단계에서 같은 이름으로 컬럼을 사용할 수 있도록 sender/receiver 기준 컬럼을 정리합니다. 원본 컬럼은 검증 가능성을 위해 가능한 한 보존합니다.

In [5]:
# =========================
# 컬럼 탐지
# =========================

# 원본 컬럼명이 달라도 표준 컬럼으로 연결될 수 있는지 확인합니다.
def validate_required_columns(df):
    cols = df.columns
    detected = {
        "timestamp_col": detect_column(cols, CONFIG["TIMESTAMP_COL_CANDIDATES"], True, "timestamp"),
        "label_col": detect_column(cols, CONFIG["LABEL_COL_CANDIDATES"], True, "label"),
        "sender_col": detect_column(cols, CONFIG["SENDER_COL_CANDIDATES"], True, "sender account"),
        "receiver_col": detect_column(cols, CONFIG["RECEIVER_COL_CANDIDATES"], True, "receiver account"),
        "amount_col": detect_column(cols, CONFIG["AMOUNT_COL_CANDIDATES"], True, "amount"),
        "from_bank_col": detect_column(cols, CONFIG["FROM_BANK_COL_CANDIDATES"], False, "from bank"),
        "to_bank_col": detect_column(cols, CONFIG["TO_BANK_COL_CANDIDATES"], False, "to bank"),
        "payment_format_col": detect_column(cols, CONFIG["PAYMENT_FORMAT_COL_CANDIDATES"], False, "payment format"),
        "tx_id_col": detect_column(cols, CONFIG["TX_ID_COL_CANDIDATES"], False, "tx_id"),
    }

    amount_keys = {normalize_name(c) for c in CONFIG["AMOUNT_COL_CANDIDATES"]}
    currency_keys = {normalize_name(c) for c in CONFIG["CURRENCY_COL_CANDIDATES"]}
    detected["amount_cols_all"] = [c for c in cols if normalize_name(c) in amount_keys]
    detected["currency_cols_all"] = [c for c in cols if normalize_name(c) in currency_keys]

    if CONFIG["USE_BANK_ACCOUNT_COMPOSITE_ID"]:
        if detected["from_bank_col"] is None or detected["to_bank_col"] is None:
            raise ValueError("composite 계좌 ID를 만들려면 From Bank / To Bank 컬럼이 필요합니다.")

    add_validation("required_columns", "PASS", "P0", json.dumps(detected, ensure_ascii=False), 0)
    return detected


COLS = validate_required_columns(df_raw)
COLS


{'timestamp_col': 'Timestamp',
 'label_col': 'Is Laundering',
 'sender_col': 'Account',
 'receiver_col': 'Account.1',
 'amount_col': 'Amount Paid',
 'from_bank_col': 'From Bank',
 'to_bank_col': 'To Bank',
 'payment_format_col': 'Payment Format',
 'tx_id_col': None,
 'amount_cols_all': ['Amount Received', 'Amount Paid'],
 'currency_cols_all': ['Receiving Currency', 'Payment Currency']}

## 6. Timestamp 변환 및 검증
시간순 split과 누수 방지를 위해 Timestamp는 가장 중요한 기준 컬럼입니다. 파싱 실패가 있으면 몇 행이 문제인지 리포트에 남깁니다.

In [6]:
# =========================
# schema review
# =========================

# 변환 전후 컬럼 타입과 결측 상태를 비교할 수 있도록 리포트 행을 만듭니다.
def make_schema_review(df, detected_cols):
    n_rows = df.height
    check_cols = {
        "source_row_number",
        detected_cols["timestamp_col"],
        detected_cols["label_col"],
        detected_cols["sender_col"],
        detected_cols["receiver_col"],
        detected_cols["amount_col"],
    }
    for k in ["from_bank_col", "to_bank_col", "tx_id_col", "payment_format_col"]:
        if detected_cols.get(k):
            check_cols.add(detected_cols[k])
    check_cols.update(detected_cols.get("amount_cols_all", []))
    check_cols.update(detected_cols.get("currency_cols_all", []))

    rows = []
    for col in df.columns:
        null_count = df.select(pl.col(col).null_count()).item()
        unique_count = None
        if col in check_cols:
            try:
                unique_count = df.select(pl.col(col).n_unique()).item()
            except Exception:
                unique_count = None
        rows.append({
            "record_type": "schema",
            "column_name": col,
            "dtype": str(df.schema[col]),
            "null_count": int(null_count),
            "null_ratio": float(null_count / n_rows) if n_rows else None,
            "unique_count": unique_count,
            "check_name": None,
            "status": None,
            "severity": None,
            "details": None,
            "n_failed_rows": None,
            "created_at": _now(),
        })
    return pl.DataFrame(rows)


schema_review_before = make_schema_review(df_raw, COLS)
schema_review_before.head(20)


record_type,column_name,dtype,null_count,null_ratio,unique_count,check_name,status,severity,details,n_failed_rows,created_at
str,str,str,i64,f64,i64,null,null,null,null,null,str
"""schema""","""source_row_number""","""UInt32""",0,0.0,5078345,null,null,null,null,null,"""2026-05-08 05:42:33"""
"""schema""","""Timestamp""","""String""",0,0.0,15018,null,null,null,null,null,"""2026-05-08 05:42:33"""
"""schema""","""From Bank""","""String""",0,0.0,30528,null,null,null,null,null,"""2026-05-08 05:42:33"""
"""schema""","""Account""","""String""",0,0.0,496995,null,null,null,null,null,"""2026-05-08 05:42:33"""
"""schema""","""To Bank""","""String""",0,0.0,15850,null,null,null,null,null,"""2026-05-08 05:42:33"""
…,…,…,…,…,…,…,…,…,…,…,…
"""schema""","""Receiving Currency""","""String""",0,0.0,15,null,null,null,null,null,"""2026-05-08 05:42:34"""
"""schema""","""Amount Paid""","""Float64""",0,0.0,923873,null,null,null,null,null,"""2026-05-08 05:42:34"""
"""schema""","""Payment Currency""","""String""",0,0.0,15,null,null,null,null,null,"""2026-05-08 05:42:34"""


## 7. Label 검증
`Is_Laundering`은 예측 타겟이므로 0/1 이외 값이나 결측치를 임의로 0으로 채우지 않습니다. 문제가 있으면 기본적으로 중단하는 것이 안전합니다.

In [7]:
# =========================
# 핵심 컬럼 정리
# =========================

# 여러 timestamp 포맷을 순서대로 시도해서 파싱 성공률을 높입니다.
def parse_timestamp_expr(col_name):
    raw = pl.col(col_name).cast(pl.Utf8).str.strip_chars()
    parsed = [raw.str.strptime(pl.Datetime, format=fmt, strict=False) for fmt in CONFIG["TIMESTAMP_FORMATS"]]
    return pl.coalesce(parsed)


def account_id_expr(bank_col, account_col):
    bank = pl.col(bank_col).cast(pl.Utf8).str.strip_chars()
    acct = pl.col(account_col).cast(pl.Utf8).str.strip_chars()
    return pl.concat_str([bank, pl.lit("|"), acct])


# 원본 컬럼을 프로젝트 표준 컬럼명으로 정리합니다.
def cast_core_columns(df, detected):
    ts_col = detected["timestamp_col"]
    label_col = detected["label_col"]
    sender_col = detected["sender_col"]
    receiver_col = detected["receiver_col"]
    amount_cols = detected.get("amount_cols_all") or [detected["amount_col"]]

    exprs = [
        pl.col(ts_col).cast(pl.Utf8).str.strip_chars().alias("timestamp_raw"),
        parse_timestamp_expr(ts_col).alias("timestamp"),
        pl.col(label_col).cast(pl.Int64, strict=False).alias("is_laundering"),
    ]

    if CONFIG["USE_BANK_ACCOUNT_COMPOSITE_ID"]:
        exprs.extend([
            account_id_expr(detected["from_bank_col"], sender_col).alias("sender_account_id"),
            account_id_expr(detected["to_bank_col"], receiver_col).alias("receiver_account_id"),
            pl.col(detected["from_bank_col"]).cast(pl.Utf8).str.strip_chars().alias("sender_bank_id"),
            pl.col(detected["to_bank_col"]).cast(pl.Utf8).str.strip_chars().alias("receiver_bank_id"),
        ])
    else:
        exprs.extend([
            pl.col(sender_col).cast(pl.Utf8).str.strip_chars().alias("sender_account_id"),
            pl.col(receiver_col).cast(pl.Utf8).str.strip_chars().alias("receiver_account_id"),
        ])

    for col in amount_cols:
        exprs.append(pl.col(col).cast(pl.Float64, strict=False).alias(compact_col_name(col)))

    exprs.append(pl.col(detected["amount_col"]).cast(pl.Float64, strict=False).alias("amount"))

    out = df.with_columns(exprs)
    out = out.with_columns(pl.col("is_laundering").cast(pl.Int8, strict=False))
    return out


df = cast_core_columns(df_raw, COLS)
profile_memory(df, "02_core_columns")

df.select([
    "source_row_number", "timestamp_raw", "timestamp",
    "sender_account_id", "receiver_account_id", "amount", "is_laundering"
]).head(5)


[02_core_columns] rows=5,078,345, cols=22, memory=910.97 MB


source_row_number,timestamp_raw,timestamp,sender_account_id,receiver_account_id,amount,is_laundering
u32,str,datetime[μs],str,str,f64,i8
0,"""2022/09/01 00:20""",2022-09-01 00:20:00,"""010|8000EBD30""","""010|8000EBD30""",3697.34,0
1,"""2022/09/01 00:20""",2022-09-01 00:20:00,"""03208|8000F4580""","""001|8000F5340""",0.01,0
2,"""2022/09/01 00:00""",2022-09-01 00:00:00,"""03209|8000F4670""","""03209|8000F4670""",14675.57,0
3,"""2022/09/01 00:02""",2022-09-01 00:02:00,"""012|8000F5030""","""012|8000F5030""",2806.97,0
4,"""2022/09/01 00:06""",2022-09-01 00:06:00,"""010|8000F5200""","""010|8000F5200""",36682.97,0


## 8. 거래 ID와 계좌 ID 정리
거래 단위 예측을 위해 `tx_id`를 만들고, 송신/수신 계좌를 일관된 account id로 정리합니다. 이 매핑은 이후 GNN에서도 사용됩니다.

In [8]:
# =========================
# timestamp 검증 및 정렬
# =========================

# timestamp 파싱 실패 행을 확인하고 실패가 있으면 중단합니다.
def validate_timestamp_parse(df):
    total = df.height
    raw_null = df.select(pl.col("timestamp_raw").is_null().sum()).item()
    parsed_null = df.select(pl.col("timestamp").is_null().sum()).item()

    if parsed_null == total:
        add_validation("timestamp_parse", "FAIL", "P0", "all timestamp values failed to parse", parsed_null)
        raise ValueError("timestamp parsing이 전부 실패했습니다. TIMESTAMP_FORMATS를 확인하세요.")

    status = "PASS" if parsed_null == 0 else "WARN"
    add_validation(
        "timestamp_parse", status, "P0" if parsed_null else "INFO",
        f"raw_null={raw_null}, parsed_null={parsed_null}, total={total}",
        parsed_null,
    )
    print(f"timestamp parsed_null={parsed_null:,} / total={total:,}")
    return parsed_null


bad_ts = validate_timestamp_parse(df)
if bad_ts > 0:
    print(f"timestamp가 없는 row {bad_ts:,}건 제외")
    df = df.filter(pl.col("timestamp").is_not_null())

df = df.sort(["timestamp", "source_row_number"])

print("timestamp min:", df.select(pl.col("timestamp").min()).item())
print("timestamp max:", df.select(pl.col("timestamp").max()).item())
profile_memory(df, "03_timestamp_sorted")


timestamp parsed_null=0 / total=5,078,345
timestamp min: 2022-09-01 00:00:00
timestamp max: 2022-09-18 16:18:00
[03_timestamp_sorted] rows=5,078,345, cols=22, memory=910.36 MB


## 9. 공통 clean base 생성
ML/GNN이 공통으로 사용할 기준 테이블을 만듭니다. 이 파일에는 아직 ML history feature나 PageRank 같은 파생 피처를 넣지 않았습니다.

In [9]:
# =========================
# label 검증
# =========================

# 라벨이 0/1로만 구성되어 있는지 검증합니다.
def validate_label_binary(df, strict=True):
    total = df.height
    invalid = df.filter(
        pl.col("is_laundering").is_null() |
        (~pl.col("is_laundering").is_in([0, 1]))
    )
    invalid_count = invalid.height
    invalid_ratio = invalid_count / total if total else 0.0

    rows = [
        {"report_type": "summary", "metric": "total_rows", "value": total, "details": None},
        {"report_type": "summary", "metric": "invalid_label_rows", "value": invalid_count, "details": f"ratio={invalid_ratio:.8f}"},
        {"report_type": "summary", "metric": "strict_label_check", "value": int(strict), "details": None},
    ]

    dist = df.group_by("is_laundering").agg(pl.count().alias("row_count")).sort("is_laundering")
    for r in dist.to_dicts():
        rows.append({
            "report_type": "label_distribution",
            "metric": f"is_laundering={r['is_laundering']}",
            "value": r["row_count"],
            "details": None,
        })

    if invalid_count > 0:
        keep_cols = [
            "source_row_number", "timestamp_raw", "timestamp",
            "sender_account_id", "receiver_account_id", "amount", "is_laundering"
        ]
        for i, r in enumerate(invalid.select([c for c in keep_cols if c in invalid.columns]).head(20).to_dicts()):
            rows.append({
                "report_type": "invalid_example",
                "metric": f"example_{i}",
                "value": None,
                "details": json.dumps(r, ensure_ascii=False, default=str),
            })

    report = pl.DataFrame(rows)
    path = CONFIG["LOCAL_OUTPUT_DIR"] / "label_verification_report.csv"
    write_csv(report, path)
    print("saved:", path)

    if invalid_count > 0:
        add_validation("label_binary", "FAIL", "P0", f"invalid_count={invalid_count}", invalid_count)
        if strict:
            raise ValueError("label 값이 0/1이 아닌 row가 있습니다. label_verification_report.csv를 확인하세요.")
        print(f"label 오류 row {invalid_count:,}건 제외")
        return df.filter(pl.col("is_laundering").is_in([0, 1]))

    add_validation("label_binary", "PASS", "P0", "all labels are 0/1", 0)
    return df


df = validate_label_binary(df, CONFIG["STRICT_LABEL_CHECK"])
profile_memory(df, "04_label_checked")

df.group_by("is_laundering").agg(pl.count().alias("row_count")).sort("is_laundering")


saved: /content/GraphAML_local_outputs/step01_clean_base/label_verification_report.csv
[04_label_checked] rows=5,078,345, cols=22, memory=910.36 MB


is_laundering,row_count
i8,u32
0,5073168
1,5177


## 10. account_mapping 생성
계좌 문자열을 모델과 그래프에서 쓰기 쉬운 정수형 ID로 매핑합니다. GNN 단계에서 새로 만들지 않고 이 매핑을 사용하기 위함입니다.

In [10]:
# =========================
# tx_id 생성
# =========================

# 원본 거래 ID가 없을 때도 재현 가능한 tx_id를 생성합니다.
def create_tx_id(df, detected):
    tx_col = detected.get("tx_id_col")
    if tx_col:
        null_cnt = df.select(pl.col(tx_col).is_null().sum()).item()
        uniq_cnt = df.select(pl.col(tx_col).n_unique()).item()
        dup_cnt = df.height - uniq_cnt
        if null_cnt == 0 and dup_cnt == 0:
            print(f"기존 거래 ID 사용: {tx_col}")
            return df.with_columns(pl.col(tx_col).cast(pl.Utf8).alias("tx_id"))
        print(f"기존 거래 ID 사용 안 함: null={null_cnt}, dup={dup_cnt}")

    # 정렬 이후 순번. 이후 파일 병합 기준으로만 사용.
    return df.with_row_index("tx_id")


def validate_tx_id_unique(df):
    null_cnt = df.select(pl.col("tx_id").is_null().sum()).item()
    uniq_cnt = df.select(pl.col("tx_id").n_unique()).item()
    dup_cnt = df.height - uniq_cnt

    if null_cnt or dup_cnt:
        add_validation("tx_id_unique", "FAIL", "P0", f"null={null_cnt}, dup={dup_cnt}", null_cnt + dup_cnt)
        raise AssertionError("tx_id에 null 또는 중복이 있습니다.")

    add_validation("tx_id_unique", "PASS", "P0", f"unique={uniq_cnt}", 0)
    print("tx_id check: PASS")


# 이미 tx_id가 있으면 중복 생성 방지
if "tx_id" in df.columns:
    df = df.drop("tx_id")

# 이후 ML/GNN 산출물 연결을 위해 tx_id를 확정합니다.
df = create_tx_id(df, COLS)
validate_tx_id_unique(df)

df.select(["tx_id", "source_row_number", "timestamp", "sender_account_id", "receiver_account_id"]).head(5)


tx_id check: PASS


tx_id,source_row_number,timestamp,sender_account_id,receiver_account_id
u32,u32,datetime[μs],str,str
0,2,2022-09-01 00:00:00,"""03209|8000F4670""","""03209|8000F4670"""
1,17,2022-09-01 00:00:00,"""01420|8005DFEB0""","""01420|8005DFEB0"""
2,158,2022-09-01 00:00:00,"""010|8000F6850""","""010|8000F6850"""
3,218,2022-09-01 00:00:00,"""012|8001167D0""","""012|8001167D0"""
4,281,2022-09-01 00:00:00,"""001|8001364A0""","""001|8001364A0"""


## 11. 스키마/라벨/메모리 리포트 저장
데이터 타입, 삭제 row, 라벨 검증, 메모리 사용량을 리포트로 남깁니다.

In [11]:
# =========================
# account_mapping 생성
# =========================

# sender/receiver 계좌를 하나의 노드 ID 체계로 매핑합니다.
def create_account_mapping(df):
    sender_null = df.select(pl.col("sender_account_id").is_null().sum()).item()
    receiver_null = df.select(pl.col("receiver_account_id").is_null().sum()).item()

    if sender_null or receiver_null:
        add_validation("account_id_null", "FAIL", "P0", f"sender_null={sender_null}, receiver_null={receiver_null}", sender_null + receiver_null)
        raise ValueError("sender/receiver 계좌 ID에 null이 있습니다.")

    accounts = pl.concat([
        df.select(pl.col("sender_account_id").alias("account_id")),
        df.select(pl.col("receiver_account_id").alias("account_id")),
    ])

    mapping = (
        accounts
        .unique()
        .sort("account_id")
        .with_row_index("node_idx")
        .select(["account_id", "node_idx"])
    )

    path = CONFIG["OUTPUT_DIR"] / "account_mapping.csv"
    write_csv(mapping, path)
    print("saved:", path)
    return mapping


def validate_account_mapping(df, mapping):
    null_cnt = mapping.select(pl.col("account_id").is_null().sum()).item()
    dup_cnt = mapping.height - mapping.select(pl.col("account_id").n_unique()).item()
    if null_cnt or dup_cnt:
        add_validation("account_mapping_unique", "FAIL", "P0", f"null={null_cnt}, dup={dup_cnt}", null_cnt + dup_cnt)
        raise AssertionError("account_mapping에 null 또는 중복이 있습니다.")

    sender_missing = (
        df.select(pl.col("sender_account_id").alias("account_id"))
        .join(mapping, on="account_id", how="left")
        .filter(pl.col("node_idx").is_null())
        .height
    )
    receiver_missing = (
        df.select(pl.col("receiver_account_id").alias("account_id"))
        .join(mapping, on="account_id", how="left")
        .filter(pl.col("node_idx").is_null())
        .height
    )

    if sender_missing or receiver_missing:
        add_validation("account_mapping_coverage", "FAIL", "P0", f"sender_missing={sender_missing}, receiver_missing={receiver_missing}", sender_missing + receiver_missing)
        raise AssertionError("account_mapping 누락이 있습니다.")

    add_validation("account_mapping_coverage", "PASS", "P0", f"n_accounts={mapping.height}", 0)
    print(f"account_mapping check: PASS, accounts={mapping.height:,}")


# GNN과 ML 모두 같은 계좌 매핑을 쓰도록 여기서 만듭니다.
account_mapping = create_account_mapping(df)
validate_account_mapping(df, account_mapping)
account_mapping.head(5)


saved: /content/drive/MyDrive/GraphAML/data/processed/step01_clean_base/account_mapping.csv
account_mapping check: PASS, accounts=515,088


account_id,node_idx
str,u32
"""0010057|801FB1090""",0
"""0010057|803A115E0""",1
"""0010057|803AA8E90""",2
"""0010057|803AAB430""",3
"""0010057|803AACE20""",4


## 12. clean_base 저장
후속 ML/GNN 노트북이 읽을 공통 기준 파일을 저장합니다. 이 산출물이 이후 파이프라인의 입력값이 됩니다.

In [12]:
# =========================
# 최종 검증
# =========================

# 저장 전 필수 컬럼과 mapping 정합성을 마지막으로 점검합니다.
# 최종 저장 전에 필수 검증을 통과했는지 확인합니다.
def run_all_validations(df, account_mapping):
    required = [
        "tx_id", "source_row_number", "timestamp", "timestamp_raw",
        "sender_account_id", "receiver_account_id", "amount", "is_laundering",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        add_validation("clean_base_required_columns", "FAIL", "P0", f"missing={missing}", 0)
        raise AssertionError(f"clean_base 필수 컬럼 누락: {missing}")
    add_validation("clean_base_required_columns", "PASS", "P0", f"required={required}", 0)

    validate_timestamp_parse(df)
    validate_tx_id_unique(df)
    validate_account_mapping(df, account_mapping)

    bad_label = df.filter(
        pl.col("is_laundering").is_null() |
        (~pl.col("is_laundering").is_in([0, 1]))
    ).height
    if bad_label:
        add_validation("final_label_check", "FAIL", "P0", f"bad_label={bad_label}", bad_label)
        raise AssertionError("최종 데이터에 label 오류가 남아 있습니다.")
    add_validation("final_label_check", "PASS", "P0", "labels are 0/1", 0)

    print("final validation: PASS")


# 최종 저장 전에 필수 검증을 통과했는지 확인합니다.
run_all_validations(df, account_mapping)


timestamp parsed_null=0 / total=5,078,345
tx_id check: PASS
account_mapping check: PASS, accounts=515,088
final validation: PASS


## 13. 최종 확인 출력
생성된 파일과 주요 row 수를 화면에 출력합니다. 실행 후 이 셀을 보고 정상 완료 여부를 빠르게 확인하면 됩니다.

In [13]:
# =========================
# clean_base 저장
# =========================

front_cols = [
    "tx_id", "source_row_number", "timestamp", "timestamp_raw",
    "sender_account_id", "receiver_account_id", "sender_bank_id", "receiver_bank_id",
    "amount", "is_laundering",
]
front_cols = [c for c in front_cols if c in df.columns]
other_cols = [c for c in df.columns if c not in front_cols]

df_clean = df.select(front_cols + other_cols)

clean_base_path = CONFIG["OUTPUT_DIR"] / "clean_base.parquet"
df_clean.write_parquet(clean_base_path)
print("saved:", clean_base_path)
profile_memory(df_clean, "05_clean_base_saved")

df_clean.head(5)


saved: /content/drive/MyDrive/GraphAML/data/processed/step01_clean_base/clean_base.parquet
[05_clean_base_saved] rows=5,078,345, cols=23, memory=929.74 MB


tx_id,source_row_number,timestamp,timestamp_raw,sender_account_id,receiver_account_id,sender_bank_id,receiver_bank_id,amount,is_laundering,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,amount_received,amount_paid
u32,u32,datetime[μs],str,str,str,str,str,f64,i8,str,str,str,str,str,f64,str,f64,str,str,i64,f64,f64
0,2,2022-09-01 00:00:00,"""2022/09/01 00:00""","""03209|8000F4670""","""03209|8000F4670""","""03209""","""03209""",14675.57,0,"""2022/09/01 00:00""","""03209""","""8000F4670""","""03209""","""8000F4670""",14675.57,"""US Dollar""",14675.57,"""US Dollar""","""Reinvestment""",0,14675.57,14675.57
1,17,2022-09-01 00:00:00,"""2022/09/01 00:00""","""01420|8005DFEB0""","""01420|8005DFEB0""","""01420""","""01420""",897.37,0,"""2022/09/01 00:00""","""01420""","""8005DFEB0""","""01420""","""8005DFEB0""",897.37,"""US Dollar""",897.37,"""US Dollar""","""Reinvestment""",0,897.37,897.37
2,158,2022-09-01 00:00:00,"""2022/09/01 00:00""","""010|8000F6850""","""010|8000F6850""","""010""","""010""",99986.94,0,"""2022/09/01 00:00""","""010""","""8000F6850""","""010""","""8000F6850""",99986.94,"""US Dollar""",99986.94,"""US Dollar""","""Reinvestment""",0,99986.94,99986.94
3,218,2022-09-01 00:00:00,"""2022/09/01 00:00""","""012|8001167D0""","""012|8001167D0""","""012""","""012""",16.08,0,"""2022/09/01 00:00""","""012""","""8001167D0""","""012""","""8001167D0""",16.08,"""US Dollar""",16.08,"""US Dollar""","""Reinvestment""",0,16.08,16.08
4,281,2022-09-01 00:00:00,"""2022/09/01 00:00""","""001|8001364A0""","""001|8001364A0""","""001""","""001""",10.3,0,"""2022/09/01 00:00""","""001""","""8001364A0""","""001""","""8001364A0""",10.3,"""US Dollar""",10.3,"""US Dollar""","""Reinvestment""",0,10.3,10.3


In [14]:
# ============================================================
# 최종 확인
# ============================================================
# 이 단계의 산출물은 clean_base와 account_mapping만 확인합니다.
# label 검증표는 LOCAL_OUTPUT_DIR에 저장하고, clean_base/account_mapping은 Drive에 저장합니다.

outputs = {
    "clean_base.parquet": CONFIG["DRIVE_OUTPUT_DIR"] / "clean_base.parquet",
    "account_mapping.csv": CONFIG["DRIVE_OUTPUT_DIR"] / "account_mapping.csv",
    "label_verification_report.csv": CONFIG["LOCAL_OUTPUT_DIR"] / "label_verification_report.csv",
}

print("생성 파일")
for name, path in outputs.items():
    print(f"- {name}: {path} / exists={path.exists()}")

print("\nclean_base shape:", df_clean.shape)
print("account_mapping shape:", account_mapping.shape)

print("\ntimestamp range")
print(df_clean.select([
    pl.col("timestamp").min().alias("min"),
    pl.col("timestamp").max().alias("max"),
]))

print("\nlabel distribution")
print(
    df_clean.group_by("is_laundering")
    .agg(pl.len().alias("row_count"))
    .with_columns((pl.col("row_count") / pl.col("row_count").sum()).alias("ratio"))
    .sort("is_laundering")
)

print("\npreprocessing check: PASS")

생성 파일
- clean_base.parquet: /content/drive/MyDrive/GraphAML/data/processed/step01_clean_base/clean_base.parquet / exists=True
- account_mapping.csv: /content/drive/MyDrive/GraphAML/data/processed/step01_clean_base/account_mapping.csv / exists=True
- label_verification_report.csv: /content/GraphAML_local_outputs/step01_clean_base/label_verification_report.csv / exists=True

clean_base shape: (5078345, 23)
account_mapping shape: (515088, 2)

timestamp range
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min                 ┆ max                 │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2022-09-01 00:00:00 ┆ 2022-09-18 16:18:00 │
└─────────────────────┴─────────────────────┘

label distribution
shape: (2, 3)
┌───────────────┬───────────┬──────────┐
│ is_laundering ┆ row_count ┆ ratio    │
│ ---           ┆ ---       ┆ ---      │
│ i8            ┆ u32       ┆ f64      │
╞═════

In [ ]:
# ============================================================
# Download helper: Step 01 clean/base lightweight reports
# - Colab local output을 zip으로 묶어 로컬 PC에 다운로드
# - 필요 시 Drive에도 백업 복사
# ============================================================

from pathlib import Path
import shutil

# 02 노트북의 가벼운 산출물 위치
src_dir = Path("/content/GraphAML_local_outputs/step01_clean_base")

# zip 생성 위치: Colab runtime local
zip_base = Path("/content/step01_clean_base_reports")
zip_path = Path(str(zip_base) + ".zip")

print("local report dir:", src_dir)
print("exists:", src_dir.exists())

if not src_dir.exists():
    raise FileNotFoundError(f"Local report directory not found: {src_dir}")

print("\nfiles:")
for f in sorted(src_dir.glob("*")):
    print("-", f.name)

# 기존 zip이 있으면 삭제 후 재생성
if zip_path.exists():
    zip_path.unlink()

created_zip = shutil.make_archive(str(zip_base), "zip", root_dir=src_dir)
print("\ncreated zip:", created_zip)

# Drive에도 백업 복사
drive_backup_dir = Path("/content/drive/MyDrive/GraphAML/data/processed/step01_clean_base")
drive_backup_dir.mkdir(parents=True, exist_ok=True)

backup_zip = drive_backup_dir / zip_path.name
shutil.copy2(zip_path, backup_zip)
print("copied to Drive:", backup_zip)

# 브라우저 다운로드
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("\n자동 다운로드를 실행할 수 없습니다.")
    print("Colab 왼쪽 파일 탭에서 직접 다운로드하세요:", zip_path)
    print("또는 Drive 백업 위치에서 다운로드하세요:", backup_zip)
    print("error:", e)